## **Preprocessing**

Author: Ophélie Rutschmann

This notebook performs preprocessing of spatial transcriptomic data and saves it in a new hd5a file for further analysis. 

Before performing further analysis (such as differential gene expression analysis or deconvoloution), we need to filter and standardize the gene expression data to remove empty bins and ensure that values are comparable across bins. 

We will first perform **filtering** to remove low quality bins (empty bins, too few UMIs, too little genes expressed). 

Then, we will apply **normalization** to account for differences in total counts per cell. This rescales each cell so that they have the same total expression, so that differences are due to biology rather than sequencing depth. Then, we will **log-transform** the data to reduce the impact of highly expressed genes. 

We will work with the filtered data set output from spaceranger. We will apply our preprocessing steps on both the binned and segmented outputs.

In [1]:
import pandas as pd
import scanpy as sc

### **1. Loading the dataset**


In [2]:
# Load data 
adata_A1_binned = sc.read_10x_h5( "A1_NABHUG14_JAN26_merged_spaceranger/outs/binned_outputs/square_008um/filtered_feature_bc_matrix.h5" )
adata_D1_binned = sc.read_10x_h5( "D1_NABHUG03_JAN26_merged_spaceranger/outs/binned_outputs/square_008um/filtered_feature_bc_matrix.h5" )

/home/ruop/miniconda3/envs/spatial/lib/python3.14/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/ruop/miniconda3/envs/spatial/lib/python3.14/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [3]:
adata_A1_segment = sc.read_10x_h5( "A1_NABHUG14_JAN26_merged_spaceranger/outs/segmented_outputs/filtered_feature_cell_matrix.h5" )
adata_D1_segment = sc.read_10x_h5( "D1_NABHUG03_JAN26_merged_spaceranger/outs/segmented_outputs/filtered_feature_cell_matrix.h5" )

/home/ruop/miniconda3/envs/spatial/lib/python3.14/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/home/ruop/miniconda3/envs/spatial/lib/python3.14/site-packages/anndata/_core/anndata.py:1825: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [4]:
# Make the variable names unique
adata_A1_binned.var_names_make_unique()
adata_D1_binned.var_names_make_unique()

adata_A1_segment.var_names_make_unique()
adata_D1_segment.var_names_make_unique()

### **2. Filtering**
Due to the particularities of this data set (very few UMIs per bin), we will not filter on the number of UMIs. We will only remove completely empty bins, and bins that express less than 10 genes.

In [5]:
# Remove empty bins
sc.pp.filter_cells(adata_A1_binned, min_counts=1)
sc.pp.filter_cells(adata_D1_binned, min_counts=1)

In [6]:
sc.pp.filter_cells(adata_A1_segment, min_counts=1)
sc.pp.filter_cells(adata_D1_segment, min_counts=1)

In [8]:
# Remove bins expressing less than 10 genes. 
sc.pp.filter_cells(adata_A1_binned, min_genes=10)
sc.pp.filter_cells(adata_D1_binned, min_genes=10)

In [9]:
sc.pp.filter_cells(adata_A1_segment, min_genes=10)
sc.pp.filter_cells(adata_D1_segment, min_genes=10)

To remove noise in downstream analysis, we will also remove genes that are expressed in less than 5 bins (i.e. count > 0 in at least 5 bins).

In [10]:
sc.pp.filter_genes(adata_A1_binned, min_cells=5)
sc.pp.filter_genes(adata_D1_binned, min_cells=5)

In [11]:
sc.pp.filter_genes(adata_A1_segment, min_cells=5)
sc.pp.filter_genes(adata_D1_segment, min_cells=5)

### **3. Normalizing the data**
Let's now normalize the data. As mentioned above, this rescales the cells so that they have the same total expression. We will aslo log-transform the data to reduce the impact of highly expressed genes. 

In [12]:
# Normalization
sc.pp.normalize_total(adata_A1_binned, target_sum=1e4)
sc.pp.normalize_total(adata_D1_binned, target_sum=1e4)

In [13]:
sc.pp.normalize_total(adata_A1_segment, target_sum=1e4)
sc.pp.normalize_total(adata_D1_segment, target_sum=1e4)

In [14]:
# Log transform
sc.pp.log1p(adata_A1_binned)
sc.pp.log1p(adata_D1_binned)

In [15]:
# Log transform
sc.pp.log1p(adata_A1_segment)
sc.pp.log1p(adata_D1_segment)

### **4. Export the data**
Let's now save the processed data. 

In [17]:
adata_A1_binned.write("A1_NABHUG14_JAN26_merged_spaceranger/outs/binned_outputs/square_008um/filtered_feature_bc_matrix_filtered_normalized.h5ad")
adata_A1_segment.write("A1_NABHUG14_JAN26_merged_spaceranger/outs/segmented_outputs/filtered_feature_cell_matrix_filtered_normalized.h5ad")

adata_D1_binned.write("D1_NABHUG03_JAN26_merged_spaceranger/outs/binned_outputs/square_008um/filtered_feature_bc_matrix_filtered_normalized.h5ad")
adata_D1_segment.write("D1_NABHUG03_JAN26_merged_spaceranger/outs/segmented_outputs/filtered_feature_cell_matrix_filtered_normalized.h5ad")